In [1]:
# ================================================================
# SpendDNA | Your Wallet's Year-End Story
# AI-assisted: parsing, merchant normalisation, analytics and report
# Allowed stack: Python + Pandas + NumPy + standard library only
# ================================================================

import os
import numpy as np
import pandas as pd

# -------------------------- 0. CONFIG ----------------------------
PROJECT_TITLE = "SpendDNA"
PERSONA_NAME = "RAHUL SHARMA"
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]

CORE_CATEGORIES = [
    "Food Delivery", "Quick Commerce", "E-commerce", "Transport",
    "Cafe", "Restaurants", "Subscriptions", "Utilities", "Groceries",
    "Investments", "Fuel", "Entertainment"
]

SPECIAL_CATEGORIES = ["Personal Transfer", "Cash Withdrawal"]


def money(value):
    return "₹{:,.0f}".format(float(value))


def pct(value):
    return "{:.1f}%".format(float(value))


def section(title):
    print("\n" + "═" * 78)
    print("  " + title.upper())
    print("═" * 78)


def bar(value, maximum, width=24):
    if maximum <= 0:
        return "·" * width
    filled = int(round((float(value) / float(maximum)) * width))
    filled = max(0, min(width, filled))
    return "█" * filled + "░" * (width - filled)


def safe_share(part, total):
    return (float(part) / float(total) * 100) if total else 0.0


def hour_label(hour):
    return "{:02d}:00".format(int(hour))


# ---------------------- 1. LOAD THE DATA -------------------------

candidate_files = [
    "rahul_transactions.csv",
    "rahul_transactions(2).csv",
    "/mnt/data/rahul_transactions(2).csv"
]

csv_file = next(
    (path for path in candidate_files if os.path.exists(path)),
    None
)

if csv_file is None:
    try:
        from google.colab import files
        print("Upload the provided transaction CSV:")
        uploaded = files.upload()
        csv_file = next(
            (name for name in uploaded if name.lower().endswith(".csv")),
            None
        )
    except Exception:
        csv_file = None

if csv_file is None:
    raise FileNotFoundError(
        "CSV not found. Keep rahul_transactions.csv in the notebook "
        "folder or upload it when prompted."
    )

raw_df = pd.read_csv(csv_file)

required_columns = [
    "Date", "Time", "Description", "Type",
    "Amount", "Balance", "Mode", "Ref"
]

missing_columns = [
    col for col in required_columns if col not in raw_df.columns
]

if missing_columns:
    raise ValueError("Missing CSV columns: " + ", ".join(missing_columns))

original_rows = len(raw_df)
duplicate_count = int(raw_df.duplicated().sum())


# ---------------------- 2. TRANSACTION PARSER ---------------------

df = raw_df.drop_duplicates().copy()

# Handle all four date formats.
try:
    df["date"] = pd.to_datetime(
        df["Date"].astype(str).str.strip(),
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )
except TypeError:
    df["date"] = pd.to_datetime(
        df["Date"].astype(str).str.strip(),
        errors="coerce",
        dayfirst=True
    )

# Handle ₹, Rs., commas and plain numbers.
amount_text = (
    df["Amount"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["amount"] = pd.to_numeric(amount_text, errors="coerce")

# Standardise transaction types.
type_text = df["Type"].astype(str).str.strip().str.lower()

type_map = {
    "dr": "debit",
    "debit": "debit",
    "cr": "credit",
    "credit": "credit"
}

df["type"] = type_text.map(type_map)

# Clean missing payment modes.
df["Mode"] = (
    df["Mode"].astype("string").str.strip().replace("", np.nan)
)

# Clean balance values.
balance_text = (
    df["Balance"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["balance"] = pd.to_numeric(balance_text, errors="coerce")

# Extract hour from HH:MM.
df["hour"] = pd.to_numeric(
    df["Time"].astype(str).str.strip().str.slice(0, 2),
    errors="coerce"
)

before_validation = len(df)

df = df.dropna(
    subset=["date", "amount", "type", "hour"]
).copy()

df["hour"] = df["hour"].astype(int)

df = df[
    (df["hour"] >= 0) & (df["hour"] <= 23)
].copy()

invalid_rows_removed = before_validation - len(df)


# ---------------------- 3. VENDOR EXTRACTOR -----------------------

# Specific patterns come before broad patterns.
# No regular expressions are used.

vendor_dictionary = {

    "ATM Cash": ["ATM-WDL"],

    "Landlord/Rent": ["RENT-LANDLORD"],

    "P2P Transfer": [
        "UPI-AMAN", "UPI-ANKIT", "UPI-PRIYA",
        "UPI-VIKAS", "UPI-NEHA", "UPI-SNEHA",
        "UPI-KARAN"
    ],

    "Instamart": ["INSTAMART"],

    "Swiggy": ["SWIGGY", "BUNDL"],

    "Restaurant (Other)": [
        "UPI-RESTAURANT",
        "BANGALORE RESTAURANT",
        "ZOMATO-DINING"
    ],

    "Zomato": ["ZOMATO"],

    "Amazon Prime Video": [
        "AMAZON PRIME", "AMZN PRIME", "PRIME VIDEO"
    ],

    "Amazon": ["AMAZON", "AMZN"],

    "Flipkart": ["FLIPKART", "FKART"],

    "Myntra": ["MYNTRA"],

    "Nykaa": ["NYKAA", "FSN E-COMMERCE"],

    "Zepto": ["ZEPTO", "KIRANAKART"],

    "Blinkit": ["BLINKIT", "GROFERS"],

    "BigBasket": ["BIGBASKET", "INNOVATIVE RETAIL"],

    "DMart": ["DMART", "AVENUE SUPERMARTS"],

    "Uber": ["UBER"],

    "Ola Prime": ["OLA-PRIME"],

    "Ola": [
        "OLACABS", "OLA CABS", "OLA-",
        "OLA ELECTRIC", "ANI TECHNOLOGIES"
    ],

    "Rapido": ["RAPIDO", "ROPPEN"],

    "BMTC": ["BMTC", "TUMMOC"],

    "Starbucks": ["STARBUCKS"],

    "Third Wave Coffee": [
        "THIRDWAVE", "THIRD WAVE", "TWC INDIA"
    ],

    "Cafe Coffee Day": ["COFFEE DAY", "CCD"],

    "Truffles": ["TRUFFLES"],

    "Empire Restaurant": ["EMPIRE"],

    "Meghana Foods": ["MEGHANA"],

    "Dineout": ["DINEOUT"],

    "Netflix": ["NETFLIX"],

    "Spotify": ["SPOTIFY"],

    "Disney+ Hotstar": ["HOTSTAR", "DISNEY"],

    "Zerodha": ["ZERODHA"],

    "Groww": ["GROWW"],

    "BESCOM": ["BESCOM", "BANGALORE ELEC"],

    "BWSSB": ["BWSSB"],

    "Airtel": ["AIRTEL", "BHARTI AIRTEL"],

    "Jio": ["JIO"],

    "Vi": ["VODAFONE", "VI POSTPAID", "UPI-VI-"],

    "HP Petrol": ["HP PETROL"],

    "BPCL": ["BPCL"],

    "Indian Oil": ["INDIAN OIL", "IOC"],

    "BookMyShow": ["BOOKMYSHOW", "BMS MOVIE"],

    "BigTree Entertainment": ["BIGTREE"],

    "Star India": ["STAR INDIA"],

    "Techcrush Labs": ["TECHCRUSH"]
}


def extract_vendor(description):

    text = str(description).upper().strip()

    for canonical_vendor, keywords in vendor_dictionary.items():

        for keyword in keywords:

            if keyword in text:
                return canonical_vendor

    return "Uncategorised"


df["vendor_clean"] = df["Description"].apply(extract_vendor)


# ---------------------- 4. CATEGORY TAGGER ------------------------

category_map = {

    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Instamart": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",

    "Truffles": "Restaurants",
    "Empire Restaurant": "Restaurants",
    "Meghana Foods": "Restaurants",
    "Dineout": "Restaurants",
    "Restaurant (Other)": "Restaurants",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney+ Hotstar": "Subscriptions",
    "Amazon Prime Video": "Subscriptions",
    "Ola Prime": "Subscriptions",

    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vi": "Utilities",
    "Landlord/Rent": "Utilities",

    "DMart": "Groceries",
    "BigBasket": "Groceries",

    "Zerodha": "Investments",
    "Groww": "Investments",

    "HP Petrol": "Fuel",
    "BPCL": "Fuel",
    "Indian Oil": "Fuel",

    "BookMyShow": "Entertainment",
    "BigTree Entertainment": "Entertainment",
    "Star India": "Entertainment",

    "P2P Transfer": "Personal Transfer",
    "ATM Cash": "Cash Withdrawal",

    "Techcrush Labs": "Income"
}

df["category"] = (
    df["vendor_clean"].map(category_map).fillna("Uncategorised")
)


# ---------------------- 5. SEPARATE TRANSACTIONS ------------------

credits = df[df["type"] == "credit"].copy()

debits = df[df["type"] == "debit"].copy()

# Exclude personal transfers and cash withdrawals from
# merchant-consumption category percentages.
merchant_spend = debits[
    ~debits["category"].isin([
        "Personal Transfer",
        "Cash Withdrawal",
        "Uncategorised"
    ])
].copy()


# ---------------------- 6. DATE ANALYSIS --------------------------

df["month_num"] = df["date"].dt.month
df["month"] = df["date"].dt.strftime("%b")
df["day_of_week"] = df["date"].dt.day_name()

df["day_type"] = [
    "Weekend" if day >= 5 else "Weekday"
    for day in df["date"].dt.dayofweek
]

merchant_spend["month_num"] = merchant_spend["date"].dt.month
merchant_spend["month"] = merchant_spend["date"].dt.strftime("%b")
merchant_spend["day_of_week"] = merchant_spend["date"].dt.day_name()

merchant_spend["day_type"] = [
    "Weekend" if day >= 5 else "Weekday"
    for day in merchant_spend["date"].dt.dayofweek
]


# ---------------------- 7. EXECUTIVE METRICS ----------------------

total_credits = float(credits["amount"].sum())
total_debits = float(debits["amount"].sum())

net_change = total_credits - total_debits

savings_rate = safe_share(net_change, total_credits)

merchant_total = float(merchant_spend["amount"].sum())

unique_vendors = int(
    merchant_spend["vendor_clean"].nunique()
)

category_totals = (
    merchant_spend.groupby("category")["amount"]
    .sum()
    .reindex(CORE_CATEGORIES, fill_value=0)
    .sort_values(ascending=False)
)

category_shares = (
    category_totals / merchant_total * 100
    if merchant_total else category_totals * 0
)

vendor_totals = (
    merchant_spend.groupby("vendor_clean")["amount"]
    .agg(
        total_spend="sum",
        transactions="count"
    )
    .sort_values("total_spend", ascending=False)
)


# ---------------------- 8. MONTHLY TREND MATRIX -------------------

month_pivot = merchant_spend.pivot_table(
    values="amount",
    index="category",
    columns="month_num",
    aggfunc="sum",
    fill_value=0
)

month_pivot = month_pivot.reindex(
    index=CORE_CATEGORIES,
    columns=[1, 2, 3, 4, 5, 6],
    fill_value=0
)

month_pivot.columns = MONTHS


# ---------------------- 9. TIME-OF-DAY MATRIX ---------------------

hour_matrix = np.zeros(
    (len(CORE_CATEGORIES), 24),
    dtype=int
)

for row_index, category in enumerate(CORE_CATEGORIES):

    category_rows = merchant_spend[
        merchant_spend["category"] == category
    ]

    for hour in range(24):

        hour_matrix[row_index, hour] = int(
            (category_rows["hour"] == hour).sum()
        )


def category_peak_hour(category):

    subset = merchant_spend[
        merchant_spend["category"] == category
    ]

    if subset.empty:
        return None, 0

    counts = subset["hour"].value_counts()

    peak = int(counts.idxmax())

    return peak, int(counts.max())


# ---------------------- 10. ANOMALY DETECTION ---------------------

# Category-wise z-score, not one global z-score.

merchant_spend["category_mean"] = (
    merchant_spend.groupby("category")["amount"]
    .transform("mean")
)

merchant_spend["category_std"] = (
    merchant_spend.groupby("category")["amount"]
    .transform("std")
)

merchant_spend["z_score"] = (
    (merchant_spend["amount"] - merchant_spend["category_mean"])
    / merchant_spend["category_std"].replace(0, np.nan)
)

anomalies = (
    merchant_spend[merchant_spend["z_score"] > 2]
    .sort_values("z_score", ascending=False)
    .copy()
)


# ---------------------- 11. ARCHETYPE DETECTION -------------------

food_categories = [
    "Food Delivery",
    "Restaurants",
    "Cafe"
]

food_spend = float(
    category_totals.reindex(food_categories, fill_value=0).sum()
)

food_share = safe_share(food_spend, merchant_total)

quick_share = safe_share(
    category_totals.get("Quick Commerce", 0),
    merchant_total
)

ecommerce_share = safe_share(
    category_totals.get("E-commerce", 0),
    merchant_total
)

investment_share = safe_share(
    category_totals.get("Investments", 0),
    merchant_total
)

transport_share = safe_share(
    category_totals.get("Transport", 0),
    merchant_total
)

food_rows = merchant_spend[
    merchant_spend["category"] == "Food Delivery"
]

late_food_rows = food_rows[
    (food_rows["hour"] >= 21) |
    (food_rows["hour"] <= 1)
]

late_food_share = safe_share(
    len(late_food_rows),
    len(food_rows)
)

subscription_vendors = merchant_spend[
    merchant_spend["category"] == "Subscriptions"
]["vendor_clean"].nunique()

archetypes = []

if food_share > 25:
    archetypes.append((
        "THE FOODIE",
        "Food + Restaurants + Cafe = " + pct(food_share)
    ))

if quick_share > 15:
    archetypes.append((
        "THE QUICK COMMERCE JUNKIE",
        "Quick Commerce = " + pct(quick_share)
    ))

if ecommerce_share > 15:
    archetypes.append((
        "THE SHOPAHOLIC",
        "E-commerce = " + pct(ecommerce_share)
    ))

if investment_share > 15:
    archetypes.append((
        "THE INVESTOR",
        "Investments = " + pct(investment_share)
    ))

if late_food_share > 50:
    archetypes.append((
        "THE LATE-NIGHT SNACKER",
        "Late-night food orders = " + pct(late_food_share)
    ))

if transport_share > 10:
    archetypes.append((
        "THE CAB COMMUTER",
        "Transport = " + pct(transport_share)
    ))

if subscription_vendors >= 5:
    archetypes.append((
        "THE SUBSCRIPTION LOVER",
        str(subscription_vendors) + " subscription vendors"
    ))

if savings_rate < 10:
    archetypes.append((
        "THE YOLO SPENDER",
        "Savings rate = " + pct(savings_rate)
    ))

if savings_rate > 40:
    archetypes.append((
        "THE DISCIPLINED SAVER",
        "Savings rate = " + pct(savings_rate)
    ))

# BONUS NINTH ARCHETYPE
# THE CITY HOPPER:
# At least 3 transport vendors and transport spending >=3%.

transport_vendor_count = merchant_spend[
    merchant_spend["category"] == "Transport"
]["vendor_clean"].nunique()

if transport_vendor_count >= 3 and transport_share >= 3:

    archetypes.append((
        "THE CITY HOPPER",
        str(transport_vendor_count)
        + " transport vendors and "
        + pct(transport_share)
        + " transport share"
    ))


# ---------------------- 12. BONUS ANALYSIS ------------------------

# Weekend versus weekday spending.

day_totals = merchant_spend.groupby("day_type")["amount"].sum()

weekday_spend = float(day_totals.get("Weekday", 0))

weekend_spend = float(day_totals.get("Weekend", 0))

weekend_vs_weekday = safe_share(
    weekend_spend - weekday_spend,
    weekday_spend
)


# Three-month rolling average forecast.
# April + May + June = projected July.

forecast_rows = []

for category in CORE_CATEGORIES:

    last_three = month_pivot.loc[
        category, ["Apr", "May", "Jun"]
    ].to_numpy(dtype=float)

    forecast_rows.append((
        category,
        float(np.mean(last_three))
    ))

forecast_df = pd.DataFrame(
    forecast_rows,
    columns=["category", "forecast_july"]
)

forecast_df = forecast_df.sort_values(
    "forecast_july",
    ascending=False
)


# Vendor cleanup audit.

unmapped = (
    df[df["vendor_clean"] == "Uncategorised"]["Description"]
    .value_counts()
    .rename_axis("description")
    .reset_index(name="rows")
)


# ================================================================
#                    FINAL SPENDDNA REPORT
# ================================================================

print("\n" + "╔" + "═" * 76 + "╗")
print("║" + " " * 23 + "💳  SpendDNA  💳" + " " * 37 + "║")
print("║" + " " * 13 + "YOUR WALLET'S SIX-MONTH STORY" + " " * 34 + "║")
print("║" + " " * 17 + "RAHUL SHARMA  |  JAN–JUN 2024" + " " * 30 + "║")
print("╚" + "═" * 76 + "╝")


# ---------------------- SECTION 01 -------------------------------

section("01 | Data Quality & Transaction Parser")

print("  Source file              :", csv_file)
print("  Original rows            :", f"{original_rows:,}")
print("  Exact duplicates removed:", f"{duplicate_count:,}")
print("  Clean transactions       :", f"{len(df):,}")
print("  Invalid rows removed     :", f"{invalid_rows_removed:,}")
print("  Unparseable dates        :", int(df["date"].isna().sum()))
print("  Unparseable amounts      :", int(df["amount"].isna().sum()))

print(
    "  Date coverage            :",
    df["date"].min().strftime("%d %b %Y"),
    "to",
    df["date"].max().strftime("%d %b %Y")
)

print("  Standardised types       :", df["type"].value_counts().to_dict())
print("  Missing payment modes    :", int(df["Mode"].isna().sum()))


# ---------------------- SECTION 02 -------------------------------

section("02 | Executive Financial Snapshot")

print(f"  Total credits            : {money(total_credits):>16}")
print(f"  Total debits             : {money(total_debits):>16}")
print(f"  Net cash movement        : {money(net_change):>16}")
print(f"  Savings rate             : {pct(savings_rate):>16}")
print(f"  Merchant spending        : {money(merchant_total):>16}")

print(
    f"  Personal transfers       : "
    f"{money(debits.loc[debits['category']=='Personal Transfer', 'amount'].sum()):>16}"
)

print(
    f"  ATM withdrawals          : "
    f"{money(debits.loc[debits['category']=='Cash Withdrawal', 'amount'].sum()):>16}"
)

print(f"  Clean transactions       : {len(df):>16,}")
print(f"  Unique merchant vendors  : {unique_vendors:>16,}")

print(
    "  Financial pulse          :",
    "POSITIVE CASH FLOW" if net_change >= 0
    else "SPENDING EXCEEDS INCOME"
)

# Compare with the PDF checkpoint without changing actual results.

reference_debits = 826679

difference_pct = safe_share(
    abs(total_debits - reference_debits),
    reference_debits
)

if difference_pct > 10:

    print("\n  ⚠ DATA CHECK: Actual debit total differs from the PDF's")
    print("    illustrative checkpoint (~₹8,26,679) by", pct(difference_pct) + ".")
    print("    Verify that this is the exact instructor-provided CSV.")

else:

    print("\n  ✓ Debit total is within 10% of the PDF's illustrative checkpoint.")


# ---------------------- SECTION 03 -------------------------------

section("03 | Category DNA — Where the Money Went")

print("  Category                 Spend              Share   Visual")

max_category_spend = (
    float(category_totals.max()) if len(category_totals) else 0
)

for category, amount in category_totals.items():

    share = safe_share(amount, merchant_total)

    print(
        f"  {category:<23} {money(amount):>13}   "
        f"{share:>6.1f}%   {bar(amount, max_category_spend, 20)}"
    )

print("\n  Note: Personal transfers and ATM withdrawals are shown separately.")
print("  They are excluded from merchant-consumption category percentages.")


# ---------------------- SECTION 04 -------------------------------

section("04 | Top Merchant Hall of Fame")

print("  Merchant                  Total spend       Transactions")

for vendor, row in vendor_totals.head(10).iterrows():

    print(
        f"  {vendor:<25} {money(row['total_spend']):>14}       "
        f"{int(row['transactions']):>5}"
    )


# ---------------------- SECTION 05 -------------------------------

section("05 | Monthly Spending Pulse")

monthly_totals = month_pivot.sum(axis=0)

max_month = (
    float(monthly_totals.max()) if len(monthly_totals) else 0
)

for month in MONTHS:

    value = float(monthly_totals[month])

    print(
        f"  {month:<4} {money(value):>13}   "
        f"{bar(value, max_month, 30)}"
    )

print("\n  Category x month matrix (₹):")

print(
    "  " + f"{'Category':<22}"
    + "".join(f"{m:>10}" for m in MONTHS)
)

for category in CORE_CATEGORIES:

    values = month_pivot.loc[category]

    print(
        "  " + f"{category:<22}"
        + "".join(f"{money(values[m]):>10}" for m in MONTHS)
    )

jan_total = float(monthly_totals.get("Jan", 0))
jun_total = float(monthly_totals.get("Jun", 0))

overall_change = safe_share(
    jun_total - jan_total,
    jan_total
)

print("\n  Merchant spend change Jan → Jun:", pct(overall_change))

growth_candidates = {}

for category in CORE_CATEGORIES:

    jan_value = float(month_pivot.loc[category, "Jan"])
    jun_value = float(month_pivot.loc[category, "Jun"])

    if jan_value > 0:

        growth_candidates[category] = safe_share(
            jun_value - jan_value,
            jan_value
        )

if growth_candidates:

    fastest = max(growth_candidates, key=growth_candidates.get)
    steepest = min(growth_candidates, key=growth_candidates.get)

    print(
        "  Biggest category rise       :",
        fastest,
        pct(growth_candidates[fastest])
    )

    print(
        "  Biggest category decline    :",
        steepest,
        pct(growth_candidates[steepest])
    )


# ---------------------- SECTION 06 -------------------------------

section("06 | Time Machine — Category x Hour Heatmap")

print("  Each character represents transaction frequency in that hour.")
print("  Legend: · none | ░ low | ▒ medium | ▓ high | █ peak")

print("  Hours :", " ".join(f"{h:02d}" for h in range(24)))

heat_symbols = ["·", "░", "▒", "▓", "█"]

for row_index, category in enumerate(CORE_CATEGORIES):

    row = hour_matrix[row_index]

    row_max = int(row.max())

    if row_max == 0:

        visual = "·" * 24

    else:

        visual = ""

        for value in row:

            level = (
                min(4, int(round((value / row_max) * 4)))
                if value > 0 else 0
            )

            visual += heat_symbols[level]

    print(f"  {category:<20} {visual}")


print("\n  Category peak hours:")

for category in [
    "Food Delivery",
    "Cafe",
    "Quick Commerce"
]:

    peak, count = category_peak_hour(category)

    if peak is not None:

        print(
            f"  {category:<20} {hour_label(peak)} "
            f"({count} transactions)"
        )


# ---------------------- SECTION 07 -------------------------------

section("07 | Weekly Rhythm — Weekday vs Weekend")

print(f"  Weekday merchant spend : {money(weekday_spend)}")
print(f"  Weekend merchant spend : {money(weekend_spend)}")
print(f"  Weekend vs weekday     : {pct(weekend_vs_weekday)}")

weekday_order = merchant_spend.groupby(
    "day_of_week"
)["amount"].sum()

day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

for day in day_order:

    print(
        f"  {day:<12} {money(weekday_order.get(day, 0)):>13}  "
        f"{bar(weekday_order.get(day, 0), float(weekday_order.max()), 18)}"
    )


# ---------------------- SECTION 08 -------------------------------

section("08 | Anomaly Radar — Unusually Large Transactions")

print("  Rule: category-wise z-score > 2 (not a fraud verdict).")
print("  Flagged transactions:", len(anomalies))

if anomalies.empty:

    print("  No transactions crossed the selected threshold.")

else:

    print("  Date          Vendor                  Category             Amount       Z")

    for _, row in anomalies.head(10).iterrows():

        print(
            f"  {row['date'].strftime('%d %b %Y'):<13} "
            f"{row['vendor_clean']:<23} "
            f"{row['category']:<20} "
            f"{money(row['amount']):>10}   "
            f"{row['z_score']:>5.2f}"
        )


# ---------------------- SECTION 09 -------------------------------

section("09 | Your Spending Personality")

print("  Archetypes are assigned only when their quantitative rules are met.")

if archetypes:

    for name, evidence in archetypes:

        print("  ✦", name)
        print("      Evidence:", evidence)

else:

    print("  No archetype rules were triggered by this dataset.")


print("\n  Rule audit:")

print(f"  Foodie combined share       : {pct(food_share)}  (rule: >25%)")
print(f"  Quick Commerce share        : {pct(quick_share)}  (rule: >15%)")
print(f"  E-commerce share            : {pct(ecommerce_share)}  (rule: >15%)")
print(f"  Investment share            : {pct(investment_share)}  (rule: >15%)")
print(f"  Late-night food orders      : {pct(late_food_share)}  (rule: >50%)")
print(f"  Transport share             : {pct(transport_share)}  (rule: >10%)")
print(f"  Subscription vendors        : {subscription_vendors}  (rule: >=5)")
print(f"  Savings rate                : {pct(savings_rate)}  (YOLO: <10%; Saver: >40%)")
print(f"  Transport vendors           : {transport_vendor_count}  (City Hopper: >=3 and share >=3%)")


# ---------------------- SECTION 10 -------------------------------

section("10 | Vendor Cleanup Audit")

print("  Canonical vendor labels:", df["vendor_clean"].nunique())

print(
    "  Uncategorised rows      :",
    int((df["vendor_clean"] == "Uncategorised").sum())
)

if unmapped.empty:

    print("  ✓ All descriptions matched a vendor rule.")

else:

    print("  Review these descriptions and extend vendor_dictionary:")

    for _, row in unmapped.iterrows():

        print(
            f"  • {row['description']} ({int(row['rows'])} rows)"
        )


# ---------------------- SECTION 11 -------------------------------

section("11 | July Spend Forecast — 3-Month Rolling Average")

print("  Simple NumPy forecast using Apr, May and Jun. Not an ML prediction.")

print(f"  {'Category':<22} {'Projected July':>16}")

for _, row in forecast_df.head(12).iterrows():

    print(
        f"  {row['category']:<22} "
        f"{money(row['forecast_july']):>16}"
    )

print(
    "  Total projected merchant spend:",
    money(forecast_df["forecast_july"].sum())
)


# ---------------------- SECTION 12 -------------------------------

section("12 | Three Data-Driven Takeaways")

if len(category_totals):

    top_category = category_totals.index[0]

    top_amount = float(category_totals.iloc[0])

    print(
        f"  1. Your largest merchant category is {top_category}, at "
        f"{money(top_amount)} "
        f"({pct(safe_share(top_amount, merchant_total))} of merchant spend)."
    )


combined_discretionary = float(
    category_totals.get("Food Delivery", 0)
    + category_totals.get("Quick Commerce", 0)
    + category_totals.get("E-commerce", 0)
)

print(
    f"  2. Food delivery + quick commerce + e-commerce account for "
    f"{pct(safe_share(combined_discretionary, merchant_total))} "
    f"of merchant spend."
)

if not anomalies.empty:

    biggest_anomaly = anomalies.iloc[0]

    print(
        f"  3. Your largest statistical outlier is "
        f"{biggest_anomaly['vendor_clean']} on "
        f"{biggest_anomaly['date'].strftime('%d %b %Y')}: "
        f"{money(biggest_anomaly['amount'])} "
        f"(z = {biggest_anomaly['z_score']:.2f})."
    )

else:

    print("  3. No transaction crossed the z-score threshold of 2.")


# ---------------------- FINAL FOOTER ------------------------------

print("\n" + "═" * 78)
print("  SpendDNA | Built with Python, Pandas and NumPy")
print("  Data note: Synthetic transactions; anomaly flags are review prompts,")
print("  not fraud findings.")
print("  AI assistance disclosed. Review vendor rules and explain the logic.")
print("═" * 78)


# ---------------------- AVAILABLE DATAFRAMES ---------------------
# df              = cleaned transaction dataset
# merchant_spend  = merchant debit transactions
# month_pivot     = category x month matrix
# hour_matrix     = category x 24-hour matrix
# anomalies       = anomaly records
# forecast_df     = next-month forecast


╔════════════════════════════════════════════════════════════════════════════╗
║                       💳  SpendDNA  💳                                     ║
║             YOUR WALLET'S SIX-MONTH STORY                                  ║
║                 RAHUL SHARMA  |  JAN–JUN 2024                              ║
╚════════════════════════════════════════════════════════════════════════════╝

══════════════════════════════════════════════════════════════════════════════
  01 | DATA QUALITY & TRANSACTION PARSER
══════════════════════════════════════════════════════════════════════════════
  Source file              : rahul_transactions.csv
  Original rows            : 1,328
  Exact duplicates removed: 18
  Clean transactions       : 1,310
  Invalid rows removed     : 0
  Unparseable dates        : 0
  Unparseable amounts      : 0
  Date coverage            : 01 Jan 2024 to 30 Jun 2024
  Standardised types       : {'debit': 1304, 'credit': 6}
  Missing payment modes    : 0

══════════════